<a href="https://colab.research.google.com/github/nat-petty/MMA3001-project/blob/main/MMA3001projectNP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
MMA3001 — Combined Cycle Power Plant Output Prediction
Aim: Predict net electrical output (PE, MW) using Average Temperature (AT) (°C), Exhaust Vacuum (V) (cm Hg), Ambient Pressure (AP) (mbar), Relative Humidity (RH) (%). Compare a linear baseline with nonlinear models and validate using the five provided shuffled sheets.

Note: Five sheets are reorderings of the same 9,568 records, not 47,840 independent samples. Never concatenate them to increase the dataset size. Use Sheet1 for exploratory analysis and model selection; use all five sheets for the supplied 5×2 validation protocol. This is not chronological forecasting: no timestamps are provided. Results apply to full-load operation within the sampled conditions.



In [1]:
%pip -q install pandas odfpy scikit-learn matplotlib seaborn joblib pytest pdoc

from pathlib import Path
import os, time, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.model_selection import train_test_split, KFold, cross_validate, GridSearchCV
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from joblib import dump
SEED = 42
sns.set_theme(style='whitegrid')
DATA = Path('data/Folds5x2_pp.ods')
if not DATA.exists():
    DATA = Path('Folds5x2_pp.ods')
if not DATA.exists():
    try:
        from google.colab import files
        print('Upload Folds5x2_pp.ods')
        uploaded = files.upload()
        DATA = Path('Folds5x2_pp.ods')
    except ImportError:
        raise FileNotFoundError('Place Folds5x2_pp.ods in data/ or current directory')
assert DATA.exists(), 'Dataset not found'
OUTPUT = Path('results'); OUTPUT.mkdir(exist_ok=True)
sheets = pd.read_excel(DATA, sheet_name=None, engine='odf')
print('Sheets:', {name: df.shape for name, df in sheets.items()})
df = sheets['Sheet1'].copy()
display(df.head())

Upload Folds5x2_pp.ods


Saving Folds5x2_pp.ods to Folds5x2_pp.ods
Sheets: {'Sheet1': (9568, 5), 'Sheet2': (9568, 5), 'Sheet3': (9568, 5), 'Sheet4': (9568, 5), 'Sheet5': (9568, 5)}


,AT,V,AP,RH,PE
0,14.96,41.76,1024.07,73.17,463.26
1,25.18,62.96,1020.04,59.08,444.37
2,5.11,39.40,1012.16,92.14,488.56
3,20.86,57.32,1010.24,76.64,446.48
4,10.82,37.50,1009.23,96.62,473.90
